# Silver to Gold
### CineData Analytics
### RocketLab 2026.2

In [0]:
# Configurando ambiente
catalog = "rocketlab"

silver_schema_name = "silver"
gold_schema_name = "gold"

silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {gold_schema}"
)

print(f"Silver: {silver_schema}")
print(f"Gold: {gold_schema}")

Silver: rocketlab.silver
Gold: rocketlab.gold


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Dimensão de filmes

gold.dim_movies

In [0]:
df_info_filmes = spark.table(
    f"{silver_schema}.tb_info_filmes"
)

# Aplicando regras de negócio
janela_sk_movie = Window.orderBy("id_filme")

df_dim_movies = (
    df_info_filmes
    .withColumn(
        "sk_movie_id",
        F.row_number()
        .over(janela_sk_movie)
        .cast("bigint")
    )
)

# Selecionando colunas para dimensão de filmes
df_dim_movies = (
    df_dim_movies
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("id_filme").cast("string"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string")
    )
)

# display(
#     df_dim_movies
#     .orderBy("sk_movie_id")
#     .limit(10)
# )

# Salvando tabela na camada Gold
(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.dim_movies"
    )
)

# Validando tabela 
df_dim_movies_gold = spark.table(
    f"{gold_schema}.dim_movies"
)

print(
    f"Registros persistidos: "
    f"{df_dim_movies_gold.count()}"
)

# df_dim_movies_gold.printSchema()
# display(
#     df_dim_movies_gold
#     .orderBy("sk_movie_id")
#     .limit(10)
# )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registros persistidos: 97879


## Dimensão de gêneros

gold.dim_genres

In [0]:
df_generos = spark.table(
    f"{silver_schema}.tb_generos"
)

# Aplicando regras de negócio
df_generos_distintos = (
    df_generos
    .select(
        F.col("genero").alias("nome_genero")
    )
    .distinct()
)

# Criando a surrogate key
janela_sk_genre = Window.orderBy("nome_genero")

df_dim_genres = (
    df_generos_distintos
    .withColumn(
        "sk_genre_id",
        F.row_number()
        .over(janela_sk_genre)
        .cast("bigint")
    )
    .select(
        F.col("sk_genre_id").cast("bigint"),
        F.col("nome_genero").cast("string")
    )
)

# display(
#     df_dim_genres
#     .orderBy("sk_genre_id")
# )

# Salvando tabela na camada gold
(
    df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.dim_genres"
    )
)

# Validando tabela
df_dim_genres_gold = spark.table(
    f"{gold_schema}.dim_genres"
)

print(
    f"Registros persistidos: "
    f"{df_dim_genres_gold.count()}"
)

# df_dim_genres_gold.printSchema()
# display(
#     df_dim_genres_gold
#     .orderBy("sk_genre_id")
# )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registros persistidos: 19


## Dimensão de pessoas

gold.dim_people

In [0]:
df_pessoas_empresas = spark.table(
    f"{silver_schema}.tb_pessoas_empresas"
)

# Aplicando as regras de negócio
tipos_pessoa = [
    "Ator",
    "Diretor",
    "Roteirista"
]

df_pessoas_distintas = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade").isin(tipos_pessoa)
    )
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
)

# Criando surrogate key
janela_sk_person = Window.orderBy(
    "nome_pessoa",
    "tipo_pessoa"
)

df_dim_people = (
    df_pessoas_distintas
    .withColumn(
        "sk_person_id",
        F.row_number()
        .over(janela_sk_person)
        .cast("bigint")
    )
    .select(
        F.col("sk_person_id").cast("bigint"),
        F.col("nome_pessoa").cast("string"),
        F.col("tipo_pessoa").cast("string")
    )
)

# df_dim_people.printSchema()
# display(
#     df_dim_people
#     .orderBy("sk_person_id")
#     .limit(10)
# )

# Salvando tabela na camada gold
(
    df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.dim_people"
    )
)

# Validando tabela 
df_dim_people_gold = spark.table(
    f"{gold_schema}.dim_people"
)

print(
    f"Registros persistidos: "
    f"{df_dim_people_gold.count()}"
)

# df_dim_people_gold.printSchema()
# display(
#     df_dim_people_gold
#     .orderBy("sk_person_id")
#     .limit(10)
# )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registros persistidos: 420589


## Dimensão de produtoras

gold.dim_companies

In [0]:
df_pessoas_empresas = spark.table(
    f"{silver_schema}.tb_pessoas_empresas"
)

# Aplicando regras de negócio
df_produtoras_distintas = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade") == "Produtora"
    )
    .select(
        F.col("nome_entidade").alias("nome_produtora")
    )
    .distinct()
)

# Criando surrogate key
janela_sk_company = Window.orderBy(
    "nome_produtora"
)

df_dim_companies = (
    df_produtoras_distintas
    .withColumn(
        "sk_company_id",
        F.row_number()
        .over(janela_sk_company)
        .cast("bigint")
    )
    .select(
        F.col("sk_company_id").cast("bigint"),
        F.col("nome_produtora").cast("string")
    )
)

# display(
#     df_dim_companies
#     .orderBy("sk_company_id")
#     .limit(10)
# )

# Salvando tabela na camada gold
(
    df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.dim_companies"
    )
)

# Validando tabela
df_dim_companies_gold = spark.table(
    f"{gold_schema}.dim_companies"
)

print(
    f"Registros persistidos: "
    f"{df_dim_companies_gold.count()}"
)

# df_dim_companies_gold.printSchema()
# display(
#     df_dim_companies_gold
#     .orderBy("sk_company_id")
#     .limit(10)
# )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registros persistidos: 45840


## Dimensão de avaliações

gold.dim_reviews

In [0]:
df_avaliacoes = spark.table(
    f"{silver_schema}.tb_avaliacoes_usuarios"
)

df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

# Aplicando regras de negócio
    # Apenas avaliações com nota são consideradas no cálculo
df_reviews_agregadas = (
    df_avaliacoes
    .filter(
        F.col("nota_usuario").isNotNull()
    )
    .groupBy("id_filme")
    .agg(
        F.count("*")
        .cast("int")
        .alias("qtd_avaliacoes_usuarios"),

        F.round(
            F.avg("nota_usuario"),
            2
        )
        .cast("double")
        .alias("nota_media_usuarios")
    )
)

# Criando tabela da gold
df_dim_reviews_base = (
    df_reviews_agregadas.alias("r")
    .join(
        df_movies
        .select(
            "sk_movie_id",
            "id_filme"
        )
        .alias("m"),
        F.col("r.id_filme") == F.col("m.id_filme"),
        "inner"
    )
    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("r.qtd_avaliacoes_usuarios")
        .cast("int")
        .alias("qtd_avaliacoes_usuarios"),

        F.col("r.nota_media_usuarios")
        .cast("double")
        .alias("nota_media_usuarios")
    )
)

# Criando surrogate key
janela_sk_review = Window.orderBy(
    "sk_movie_id"
)

df_dim_reviews = (
    df_dim_reviews_base
    .withColumn(
        "sk_review_id",
        F.row_number()
        .over(janela_sk_review)
        .cast("bigint")
    )
    .select(
        F.col("sk_review_id"),
        F.col("sk_movie_id"),
        F.col("qtd_avaliacoes_usuarios"),
        F.col("nota_media_usuarios")
    )
)


# df_dim_reviews.printSchema()
# display(
#     df_dim_reviews
#     .orderBy("sk_review_id")
#     .limit(10)
# )


# Salvando tabela na camada gold
(
    df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.dim_reviews"
    )
)

# Validando tabela
df_dim_reviews_gold = spark.table(
    f"{gold_schema}.dim_reviews"
)

print(
    f"Registros persistidos: "
    f"{df_dim_reviews_gold.count()}"
)

# df_dim_reviews_gold.printSchema()
# display(
#     df_dim_reviews_gold
#     .orderBy("sk_review_id")
#     .limit(10)
# )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Registros persistidos: 26111


## ponte filme-gênero

gold.bridge_movie_genre


In [0]:
df_generos = spark.table(
    f"{silver_schema}.tb_generos"
)

df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_genres = spark.table(
    f"{gold_schema}.dim_genres"
)

df_bridge_movie_genre = (
    df_generos.alias("s")

    .join(
        df_movies
        .select(
            "sk_movie_id",
            "id_filme"
        )
        .alias("m"),
        F.col("s.id_filme") == F.col("m.id_filme"),
        "inner"
    )

    .join(
        df_genres
        .select(
            "sk_genre_id",
            "nome_genero"
        )
        .alias("g"),
        F.col("s.genero") == F.col("g.nome_genero"),
        "inner"
    )

    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("g.sk_genre_id")
        .cast("bigint")
        .alias("sk_genre_id")
    )
)

# df_bridge_movie_genre.printSchema()
# display(
#     df_bridge_movie_genre
#     .orderBy(
#         "sk_movie_id",
#         "sk_genre_id"
#     )
#     .limit(10)
# )


# Salvando tabela na camada gold
(
    df_bridge_movie_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.bridge_movie_genre"
    )
)

# Validando tabela
df_bridge_movie_genre_gold = spark.table(
    f"{gold_schema}.bridge_movie_genre"
)

print(
    f"Registros persistidos: "
    f"{df_bridge_movie_genre_gold.count()}"
)

# df_bridge_movie_genre_gold.printSchema()
# display(
#     df_bridge_movie_genre_gold
#     .orderBy(
#         "sk_movie_id",
#         "sk_genre_id"
#     )
#     .limit(10)
# )

Registros persistidos: 140243


## ponte filme-pessoa

gold.bridge_movie_person


In [0]:
df_pessoas_empresas = spark.table(
    f"{silver_schema}.tb_pessoas_empresas"
)

df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_people = spark.table(
    f"{gold_schema}.dim_people"
)

# Aplicando regras de negócio
df_relacoes_pessoas = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade").isin(
            "Ator",
            "Diretor",
            "Roteirista"
        )
    )
)

df_bridge_movie_person = (
    df_relacoes_pessoas.alias("s")

    .join(
        df_movies
        .select(
            "sk_movie_id",
            "id_filme"
        )
        .alias("m"),
        F.col("s.id_filme") == F.col("m.id_filme"),
        "inner"
    )

    .join(
        df_people
        .select(
            "sk_person_id",
            "nome_pessoa",
            "tipo_pessoa"
        )
        .alias("p"),
        (
            F.col("s.nome_entidade")
            == F.col("p.nome_pessoa")
        )
        &
        (
            F.col("s.tipo_entidade")
            == F.col("p.tipo_pessoa")
        ),
        "inner"
    )

    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("p.sk_person_id")
        .cast("bigint")
        .alias("sk_person_id")
    )
)

# display(
#     df_bridge_movie_person
#     .orderBy(
#         "sk_movie_id",
#         "sk_person_id"
#     )
#     .limit(10)
# )

# Salvando tabela na camada gold
(
    df_bridge_movie_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.bridge_movie_person"
    )
)

# Validando tabela 
df_bridge_movie_person_gold = spark.table(
    f"{gold_schema}.bridge_movie_person"
)

print(
    f"bridge_movie_person: "
    f"{df_bridge_movie_person_gold.count()} relações"
)

# df_bridge_movie_person_gold.printSchema()
# display(
#     df_bridge_movie_person_gold
#     .orderBy(
#         "sk_movie_id",
#         "sk_person_id"
#     )
#     .limit(10)
# )

bridge_movie_person: 767629 relações


## ponte filme-produtora

gold.bridge_movie_company

In [0]:
df_pessoas_empresas = spark.table(
    f"{silver_schema}.tb_pessoas_empresas"
)

df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_companies = spark.table(
    f"{gold_schema}.dim_companies"
)

# Filtrando as produtroras
df_relacoes_produtoras = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade") == "Produtora"
    )
)

df_bridge_movie_company = (
    df_relacoes_produtoras.alias("s")

    .join(
        df_movies
        .select(
            "sk_movie_id",
            "id_filme"
        )
        .alias("m"),
        F.col("s.id_filme") == F.col("m.id_filme"),
        "inner"
    )

    .join(
        df_companies
        .select(
            "sk_company_id",
            "nome_produtora"
        )
        .alias("c"),
        F.col("s.nome_entidade")
        == F.col("c.nome_produtora"),
        "inner"
    )

    .select(
        F.col("m.sk_movie_id")
        .cast("bigint")
        .alias("sk_movie_id"),

        F.col("c.sk_company_id")
        .cast("bigint")
        .alias("sk_company_id")
    )
)

# display(
#     df_bridge_movie_company
#     .orderBy(
#         "sk_movie_id",
#         "sk_company_id"
#     )
#     .limit(10)
# )

# Salvando tabela na camada gold
(
    df_bridge_movie_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.bridge_movie_company"
    )
)

# Validando tabela 
df_bridge_movie_company_gold = spark.table(
    f"{gold_schema}.bridge_movie_company"
)

print(
    f"Registros persistidos: "
    f"{df_bridge_movie_company_gold.count()}"
)

# df_bridge_movie_company_gold.printSchema()
# display(
#     df_bridge_movie_company_gold
#     .orderBy(
#         "sk_movie_id",
#         "sk_company_id"
#     )
#     .limit(10)
# )

Registros persistidos: 118876


## Fato de performance dos filmes
gold.fact_movies_performance

In [0]:
# Como as tabelas Silver podem conter múltiplos registros para um mesmo filme, as métricas são consolidadas por id_filme antes dos joins, evitando multiplicação de registros.

# Valores nulos são desconsiderados durante a consolidação.

# OBS -> Nos poucos casos em que existem múltiplos valores não nulos distintos para o mesmo atributo, utiliza-se max como critério técnico e determinístico de desempate.

In [0]:
df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_financeiro = spark.table(
    f"{silver_schema}.tb_financeiro_filmes"
)

df_metricas = spark.table(
    f"{silver_schema}.tb_metricas_engajamento"
)

# Aplicando regras de negócio
df_movies_lancados = (
    df_movies
    .filter(
        F.col("status_filme") == "Lançado"#Considerando apenas filmes lançados
    )
)

    # Consolidando as métricas financeiras 
    # Um único registro por id_filme
df_financeiro_consolidado = (
    df_financeiro
    .groupBy("id_filme")
    .agg(
        F.max("orcamento_usd")
            .alias("orcamento_usd"),

        F.max("receita_usd")
            .alias("receita_usd"),

        F.max("orcamento_brl")
            .alias("orcamento_brl"),

        F.max("receita_brl")
            .alias("receita_brl"),

        F.max("lucro_usd")
            .alias("lucro_usd"),

        F.max("lucro_brl")
            .alias("lucro_brl")
    )
)
    # Consolidando as métricas de engajamento
    # Um único registro por id_filme.
df_metricas_consolidado = (
    df_metricas
    .groupBy("id_filme")
    .agg(
        F.max("popularidade")
            .alias("popularidade"),

        F.max("nota_media_tmdb")
            .alias("nota_media_tmdb"),

        F.max("qtd_votos_tmdb")
            .alias("qtd_votos_tmdb"),

        F.max("nota_media_imdb")
            .alias("nota_media_imdb"),

        F.max("qtd_votos_imdb")
            .alias("qtd_votos_imdb")
    )
)


# Construindo tabela fato
    # LEFT JOIN para preservar todos os filmes lançados
df_fact_movies_performance = (
    df_movies_lancados.alias("m")

    .join(
        df_financeiro_consolidado.alias("f"),
        F.col("m.id_filme") == F.col("f.id_filme"),
        "left"
    )

    .join(
        df_metricas_consolidado.alias("e"),
        F.col("m.id_filme") == F.col("e.id_filme"),
        "left"
    )

    .select(
        F.col("m.sk_movie_id")
            .cast("bigint")
            .alias("sk_movie_id"),

        F.col("f.orcamento_usd")
            .cast("decimal(18,2)")
            .alias("orcamento_usd"),

        F.col("f.receita_usd")
            .cast("decimal(18,2)")
            .alias("receita_usd"),

        F.col("f.lucro_usd")
            .cast("decimal(18,2)")
            .alias("lucro_usd"),

        F.col("f.orcamento_brl")
            .cast("decimal(18,2)")
            .alias("orcamento_brl"),

        F.col("f.receita_brl")
            .cast("decimal(18,2)")
            .alias("receita_brl"),

        F.col("f.lucro_brl")
            .cast("decimal(18,2)")
            .alias("lucro_brl"),

        F.col("e.popularidade")
            .cast("double")
            .alias("popularidade"),

        F.col("e.nota_media_tmdb")
            .cast("double")
            .alias("nota_media_tmdb"),

        F.col("e.qtd_votos_tmdb")
            .cast("int")
            .alias("qtd_votos_tmdb"),

        F.col("e.nota_media_imdb")
            .cast("double")
            .alias("nota_media_imdb"),

        F.col("e.qtd_votos_imdb")
            .cast("int")
            .alias("qtd_votos_imdb")
    )
)

# display(
#     df_fact_movies_performance
#     .limit(10)
# )

# Salvando tabela na camada gold
(
    df_fact_movies_performance
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.fact_movies_performance"
    )
)

# Validando tabela
df_fact_persistida = spark.table(
    f"{gold_schema}.fact_movies_performance"
)

print(
    f"Registros persistidos: "
    f"{df_fact_persistida.count()}"
)

# df_fact_persistida.printSchema()

Registros persistidos: 96463


## Testando Star Schema

In [0]:
dim_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

dim_genres = spark.table(
    f"{gold_schema}.dim_genres"
)

dim_people = spark.table(
    f"{gold_schema}.dim_people"
)

dim_companies = spark.table(
    f"{gold_schema}.dim_companies"
)

dim_reviews = spark.table(
    f"{gold_schema}.dim_reviews"
)

fact_movies_performance = spark.table(
    f"{gold_schema}.fact_movies_performance"
)

problemas_pk = (
    dim_movies
    .filter(F.col("sk_movie_id").isNull())
    .count()
    +
    dim_genres
    .filter(F.col("sk_genre_id").isNull())
    .count()
    +
    dim_people
    .filter(F.col("sk_person_id").isNull())
    .count()
    +
    dim_companies
    .filter(F.col("sk_company_id").isNull())
    .count()
    +
    dim_reviews
    .filter(F.col("sk_review_id").isNull())
    .count()
)

qtd_filmes_lancados = (
    dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .count()
)

qtd_registros_fact = fact_movies_performance.count()

print(
    f"Star Schema validado | "
    f"PKs nulas: {problemas_pk} | "
    f"Filmes lançados: {qtd_filmes_lancados} | "
    f"Registros na fato: {qtd_registros_fact}"
)

df_teste_star_schema = (
    fact_movies_performance.alias("f")
    .join(
        dim_movies.alias("m"),
        "sk_movie_id",
        "inner"
    )
    .select(
        "m.titulo",
        "m.ano_lancamento",
        "f.receita_usd",
        "f.receita_brl",
        "f.popularidade",
        "f.nota_media_tmdb",
        "f.nota_media_imdb"
    )
    .filter(F.col("receita_usd").isNotNull())
    .orderBy(F.desc("receita_usd"))
)

display(
    df_teste_star_schema.limit(10)
)

Star Schema validado | PKs nulas: 0 | Filmes lançados: 96463 | Registros na fato: 96463


titulo,ano_lancamento,receita_usd,receita_brl,popularidade,nota_media_tmdb,nota_media_imdb
Avengers: Endgame,2019,2800000000.00,14439320000.00,91.756,8.263,8.4
Avatar: The Way of Water,2022,2320250281.00,11965298674.09,241.285,null,7.5
AVENGERS: INFINITY WAR,2018,2052415039.00,10584099114.62,154.34,8.255,8.4
spider-man: no way home,2021,1921847111.00,9910773366.72,186.065,7.99,null
The Lion King,2019,1663075401.00,8576313535.42,63.351,7.1,6.8
Top Gun: Maverick,2022,1488732821.00,7677246284.61,126.291,8.26,8.2
Barbie,2023,1428545028.00,7366863854.89,1069.34,7.279,6.8
The Super Mario Bros. Movie,2023,1355725263.00,6991339608.76,410.411,7.775,7.0
Black Panther,2018,1349926083.00,6961433817.42,43.665,7.39,7.3
Star Wars: The Last Jedi,2017,1332698830.00,6872594596.43,47.241,6.825,6.8


### Tabela gold_genai_movies_context

In [0]:
df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_fact = spark.table(
    f"{gold_schema}.fact_movies_performance"
)

df_bridge_person = spark.table(
    f"{gold_schema}.bridge_movie_person"
)

df_people = spark.table(
    f"{gold_schema}.dim_people"
)


# Aplicando regras de negócio
df_pessoas_filmes = ( # Pegando as pessoas relacionadas a cada filme
    df_bridge_person.alias("b")
    .join(
        df_people.alias("p"),
        F.col("b.sk_person_id")
        == F.col("p.sk_person_id"),
        "inner"
    )
    .select(
        F.col("b.sk_movie_id"),
        F.col("p.nome_pessoa"),
        F.col("p.tipo_pessoa")
    )
)

# Agregação dos atores em uma única string por filme.
df_atores = (
    df_pessoas_filmes
    .filter(
        F.col("tipo_pessoa") == "Ator"
    )
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set("nome_pessoa")
            )
        ).alias("atores")
    )
)

# Agregação dos diretores em uma única string por filme.
df_diretores = (
    df_pessoas_filmes
    .filter(
        F.col("tipo_pessoa") == "Diretor"
    )
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set("nome_pessoa")
            )
        ).alias("diretores")
    )
)

# Construindo a tabela
    # A tabela fato define o conjunto de filmes lançados.
    # Usei LEFT JOIN para preservar o filme 
df_contexto_base = (
    df_fact.alias("f")

    .join(
        df_movies.alias("m"),
        F.col("f.sk_movie_id")
        == F.col("m.sk_movie_id"),
        "inner"
    )

    .join(
        df_atores.alias("a"),
        F.col("f.sk_movie_id")
        == F.col("a.sk_movie_id"),
        "left"
    )

    .join(
        df_diretores.alias("d"),
        F.col("f.sk_movie_id")
        == F.col("d.sk_movie_id"),
        "left"
    )
)


# Construção do documento textual para vetorização
    # COALESCE fornece textos de fallback para campos ausentes
    # evitando que valores nulos invalidem o documento completo
    # titulo e ano_lancamento não possuem valores nulos por isso não tratei

df_gold_genai_movies_context = (
    df_contexto_base
    .select(
        F.col("m.id_filme")
            .cast("string")
            .alias("movie_id"),

        F.col("m.titulo")
            .cast("string")
            .alias("title"),

        F.concat(
            F.lit("O filme "),
            F.col("m.titulo"),
            F.lit(", lançado no ano de "),
            F.col("m.ano_lancamento").cast("string"),

            F.lit(", faturou "),
            F.coalesce(
                F.concat(
                    F.lit("US$ "),
                    F.col("f.receita_usd").cast("string")
                ),
                F.lit("receita não informada")
            ),

            F.lit(" e teve um custo de "),
            F.coalesce(
                F.concat(
                    F.lit("US$ "),
                    F.col("f.orcamento_usd").cast("string")
                ),
                F.lit("orçamento não informado")
            ),

            F.lit(". Estrelado por "),
            F.coalesce(
                F.col("a.atores"),
                F.lit("elenco não informado")
            ),

            F.lit(" e dirigido por "),
            F.coalesce(
                F.col("d.diretores"),
                F.lit("diretor não informado")
            ),

            F.lit(
                ", o filme possui a seguinte sinopse: "
            ),

            F.coalesce(
                F.col("m.sinopse"),
                F.lit("sinopse não disponível")
            ),

            F.lit(".")
        )
        .cast("string")
        .alias("llm_context_document")
    )
)


# Salvando tabela na camada gold
(
    df_gold_genai_movies_context.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{gold_schema}.gold_genai_movies_context"
    )
)


# Validando tabela 
df_genai_context_gold = spark.table(
    f"{gold_schema}.gold_genai_movies_context"
)

print(
    f"gold_genai_movies_context: "
    f"{df_genai_context_gold.count()} documentos"
)

display(
    df_genai_context_gold.limit(15)
)

gold_genai_movies_context: 96463 documentos


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Aron Von Andrian, Erika Alexander, Izzy Jones, Steven Michael-o’hara, Tedroy Newell e dirigido por Lola Atkins, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.."
1000011,Worth Your Weight in Gold,"O filme Worth Your Weight in Gold, lançado no ano de 2022, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Inês Pronto, Isabel Abreu, João Pedro Bénard, Marcello Urgeghe e dirigido por Sandro Aguilar, o filme possui a seguinte sinopse: Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.."
1000014,On va manquer !,"O filme On va manquer !, lançado no ano de 2018, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por Biyouna, Claire Chust, Gabrielle Cohen, Maxime Pambet, Mouss Zouheyri, Soulayman Rkiba e dirigido por Sabrina Ouazani, o filme possui a seguinte sinopse: sinopse não disponível."
1000030,58 Hours: The Baby Jessica Story,"O filme 58 Hours: The Baby Jessica Story, lançado no ano de 2021, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por elenco não informado e dirigido por Mark Bone, o filme possui a seguinte sinopse: sinopse não disponível."
1000054,One Hundred Years and Hope,"O filme One Hundred Years and Hope, lançado no ano de 2022, faturou receita não informada e teve um custo de orçamento não informado. Estrelado por elenco não informado e dirigido por Takashi Nishihara, o filme possui a seguinte sinopse: In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.."
1000058,Homecoming,"O filme Homecoming, lançado no ano de 2023, faturou receita não informada e teve

### Desafio de Analytics

In [0]:
# 1 - Qual é a receita total (em R$) somada de todos os filmes da base?
df_fact = spark.table(
    f"{gold_schema}.fact_movies_performance" 
)

df_receita_total = (
    df_fact
    .agg(
        F.sum("receita_brl") # sum ignora os nulos 
        .alias("receita_total_brl")
    )
)

display(df_receita_total)

receita_total_brl
837771066819.07


In [0]:
# 2 - Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.
df_fact = spark.table(
    f"{gold_schema}.fact_movies_performance"
)

df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_top_5_popularidade = (
    df_fact.alias("f")
    .join(
        df_movies.alias("m"),
        F.col("f.sk_movie_id")
        == F.col("m.sk_movie_id"),
        "inner"
    )
    .select(
        F.col("m.titulo")
            .alias("titulo"),
        F.col("f.popularidade")
            .alias("popularidade")
    )
    .orderBy(
        F.col("popularidade").desc()
    )
    .limit(5)
)

display(df_top_5_popularidade)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


In [0]:
# 3 - Quantos filmes cada gênero possui? Liste do maior para o menor volume.
df_bridge_genre = spark.table(
    f"{gold_schema}.bridge_movie_genre"
)

df_genres = spark.table(
    f"{gold_schema}.dim_genres"
)

df_filmes_por_genero = (
    df_bridge_genre.alias("b")
    .join(
        df_genres.alias("g"),
        F.col("b.sk_genre_id")
        == F.col("g.sk_genre_id"),
        "inner"
    )
    .groupBy(
        F.col("g.nome_genero")
    )
    .agg(
        F.count("b.sk_movie_id")
        .alias("qtd_filmes")
    )
    .orderBy(
        F.col("qtd_filmes").desc()
    )
)

display(df_filmes_por_genero)

nome_genero,qtd_filmes
Drama,32251
Documentary,18983
Comedy,18588
Thriller,10266
Horror,9716
Romance,7628
Action,6035
Crime,4733
Animation,4459
TV Movie,4071


In [0]:
# 4 - Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK()).
df_fact = spark.table(
    f"{gold_schema}.fact_movies_performance"
)

df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_receitas_filmes = (
    df_fact.alias("f")
    .join(
        df_movies.alias("m"),
        F.col("f.sk_movie_id") == F.col("m.sk_movie_id"),
        "inner"
    )
    .select(
        F.col("m.titulo").alias("titulo"),
        F.col("f.receita_usd").alias("receita_usd"),
        F.col("f.receita_brl").alias("receita_brl")
    )
)

# Definindo ranking de receita em usd
window_receita = Window.orderBy(
    F.col("receita_usd").desc()
)

df_ranking_receita = (
    df_receitas_filmes
    .withColumn(
        "posicao_ranking",
        F.rank().over(window_receita)
    )
)

# Pegando o top10
df_top_10_receita = (
    df_ranking_receita
    .filter(F.col("posicao_ranking") <= 10)
    .orderBy(
        F.col("posicao_ranking").asc()
    )
)

display(df_top_10_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,14439320000.00,1
Avatar: The Way of Water,2320250281.00,11965298674.09,2
AVENGERS: INFINITY WAR,2052415039.00,10584099114.62,3
spider-man: no way home,1921847111.00,9910773366.72,4
The Lion King,1663075401.00,8576313535.42,5
Top Gun: Maverick,1488732821.00,7677246284.61,6
Barbie,1428545028.00,7366863854.89,7
The Super Mario Bros. Movie,1355725263.00,6991339608.76,8
Black Panther,1349926083.00,6961433817.42,9
Star Wars: The Last Jedi,1332698830.00,6872594596.43,10


In [0]:
# 5 - Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?
df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_bridge_person = spark.table(
    f"{gold_schema}.bridge_movie_person"
)

df_people = spark.table(
    f"{gold_schema}.dim_people"
)

# Pegando a data mais recente da base
    # Apenas filmes lançados e ignoro datas futuras.
data_referencia = (
    df_movies
    .filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") <= F.current_date())
    )
    .agg(
        F.max("data_lancamento").alias("data_referencia")
    )
    .first()["data_referencia"]
)

print(f"Data de referência da base: {data_referencia}")

# Pegando filmes lançados nos últimos 2 anos
df_filmes_ultimos_2_anos = (
    df_movies
    .filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") <= F.lit(data_referencia))
        & (
            F.col("data_lancamento")
            >= F.add_months(F.lit(data_referencia), -24)
        )
    )
    .select("sk_movie_id")
)

# Participação dos atores
df_participacoes_atores = (
    df_filmes_ultimos_2_anos.alias("m")
    .join(
        df_bridge_person.alias("b"),
        F.col("m.sk_movie_id") == F.col("b.sk_movie_id"),
        "inner"
    )
    .join(
        df_people.alias("p"),
        F.col("b.sk_person_id") == F.col("p.sk_person_id"),
        "inner"
    )
    .filter(
        F.col("p.tipo_pessoa") == "Ator"
    )
)

# Contando as participações por ator
df_ator_mais_participacoes = (
    df_participacoes_atores
    .groupBy(
        F.col("p.nome_pessoa").alias("nome_ator")
    )
    .agg(
        F.countDistinct("m.sk_movie_id")
        .alias("qtd_participacoes")
    )
    .orderBy(
        F.col("qtd_participacoes").desc()
    )
    .limit(1)
)

display(df_ator_mais_participacoes)

Data de referência da base: 2026-02-19


nome_ator,qtd_participacoes
Kevin Hart,64


In [0]:
# 6 - Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?
df_movies = spark.table(
    f"{gold_schema}.dim_movies"
)

df_fact = spark.table(
    f"{gold_schema}.fact_movies_performance"
)

df_bridge_company = spark.table(
    f"{gold_schema}.bridge_movie_company"
)

df_companies = spark.table(
    f"{gold_schema}.dim_companies"
)

# Pegando a data mais recente da base
    # Apenas filmes lançados e ignoro datas futuras.
data_referencia = (
    df_movies
    .filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") <= F.current_date())
    )
    .agg(
        F.max("data_lancamento").alias("data_referencia")
    )
    .first()["data_referencia"]
)

print(f"Data de referência da base: {data_referencia}")

# Pegando filmes lançados nos últimos 5 anos
df_filmes_ultimos_5_anos = (
    df_movies
    .filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") <= F.lit(data_referencia))
        & (
            F.col("data_lancamento")
            >= F.add_months(F.lit(data_referencia), -60)
        )
    )
    .select(
        "sk_movie_id"
    )
)

# Relacionando filmes, lucro e produtoras
df_lucro_produtoras = (
    df_filmes_ultimos_5_anos.alias("m")
    .join(
        df_fact.alias("f"),
        F.col("m.sk_movie_id") == F.col("f.sk_movie_id"),
        "inner"
    )
    .join(
        df_bridge_company.alias("b"),
        F.col("m.sk_movie_id") == F.col("b.sk_movie_id"),
        "inner"
    )
    .join(
        df_companies.alias("c"),
        F.col("b.sk_company_id") == F.col("c.sk_company_id"),
        "inner"
    )
)

# Somando lucro por produtora
    # Se um filme tem mais de uma produtora o lucro é associado as produtoras, eu não divido o lucro entre elas
    # Usei lucro_usd para facilitar a comparação com o dados originais
df_produtora_maior_lucro = (
    df_lucro_produtoras
    .groupBy(
        F.col("c.nome_produtora").alias("nome_produtora")
    )
    .agg(
        F.sum("f.lucro_usd").alias("lucro_total_usd")
    )
    .orderBy(
        F.col("lucro_total_usd").desc()
    )
    .limit(1)
)

display(df_produtora_maior_lucro)

Data de referência da base: 2026-02-19


nome_produtora,lucro_total_usd
Universal Pictures,5772329679.00
